# OData

Diversas APIs disponíveis no [portal de dados abertos](https://dadosabertos.bcb.gov.br/) do Banco Central implementam o protocolo [OData](https://www.odata.org/), são centenas de APIs.

O `python-bcb` tem algumas classes que implementam APIs OData:

- `bcb.odata.api.Expectativas`: Expectativas de mercado para os indicadores macroeconômicos da Pesquisa Focus
- `bcb.odata.api.PTAX`: Dólar comercial
- `bcb.odata.api.TaxaJuros`: Taxas de juros de operações de crédito por instituição financeira
- `bcb.odata.api.IFDATA`: Dados selecionados de instituições financeiras
- `bcb.odata.api.MercadoImobiliario`: Informações do Mercado Imobiliário
- `bcb.odata.api.SPI`: Estatísticas do SPI (Sistema de Pagamentos Instantâneos)
- Veja todas as APIs implementadas em [APIs OData](api.html#apis-odata).

Para uma explicação prática das consultas ao IFData, veja [IFData](ifdata.html).

Estas APIs foram implementadas em classes por serem as mais populares, entretanto, qualquer API OData pode ser acessada através da classe `bcb.odata.api.ODataAPI` que abstrai o acesso a API a partir da URL da API que está disponível no [portal de dados abertos](https://dadosabertos.bcb.gov.br/) do Banco Central. Veja mais detalhes em [Classe ODataAPI](#classe-odataapi).

## Timeout por chamada

As APIs OData também aceitam `timeout` para serviços lentos ou consultas com muitos registros. O timeout informado no construtor é usado na descoberta do serviço, no carregamento dos metadados e como padrão das queries criadas a partir dessa API. Métodos de execução podem receber outro valor para sobrescrever o padrão apenas naquela chamada.

``` python
from bcb import Expectativas

em = Expectativas(timeout=120)
ep = em.get_endpoint("ExpectativasMercadoAnuais")

df = ep.get(timeout=60)
raw = ep.query().limit(1000).text(timeout=60)
data = ep.query().limit(1000).collect(timeout=60)
```

O mesmo parâmetro está disponível nas versões assíncronas `async_get`, `async_text` e `async_collect`.

Segue um exemplo de como acessar a API do PIX.

In [1]:
from bcb import SPI
pix = SPI()

É necessário importar e criar um objeto da classe que implementa a API, neste caso a classe `SPI`. Tendo o objeto, executar o método `describe` para visualizar os *endpoints* disponíveis na API e suas propriedades. Para obter apenas a listagem curta dos nomes, use `describe(full=False)`.

In [2]:
pix.describe()

EntitySets:

EntitySet (Endpoint): PixLiquidadosAtual
EntityType: br.gov.bcb.olinda.servico.SPI.PixLiquidadosAtual
Properties: Data<datetime>, Quantidade<int>, CanalPrimario<int>, CanalSecundario<int>, Total<float>, Media<float>

EntitySet (Endpoint): PixRemuneracaoContaPI
EntityType: br.gov.bcb.olinda.servico.SPI.PixRemuneracaoContaPI
Properties: DataBase<datetime>, BaseRemuneracaoIF<float>, ValorRemuneradoIF<float>, BaseRemuneracaoTotal<float>, ValorRemuneradoTotal<float>, SelicAnual<float>

EntitySet (Endpoint): PixInterrupcaoSPIEventos
EntityType: br.gov.bcb.olinda.servico.SPI.PixInterrupcaoSPIEventos
Properties: DataHoraInicioInt<str>, DataHoraTerminoInt<str>, Interrupcao<str>

EntitySet (Endpoint): PixInterrupcaoSPI
EntityType: br.gov.bcb.olinda.servico.SPI.PixInterrupcaoSPI
Properties: DataHoraInicioInt<str>, DataHoraTerminoInt<str>, Interrupcao<str>

EntitySet (Endpoint): PixLiquidadosIntradia
EntityType: br.gov.bcb.olinda.servico.SPI.PixLiquidadosIntradia
Properties: Horario<s

Como vemos, a API do PIX tem 4 *endpoints* (`EntitySets`). Para restringir a saída a um *endpoint*, execute o método `describe` passando como argumento o nome do *endpoint*.

In [3]:
pix.describe("PixLiquidadosAtual")


EntitySet (Endpoint): PixLiquidadosAtual
EntityType: br.gov.bcb.olinda.servico.SPI.PixLiquidadosAtual
Properties: Data<datetime>, Quantidade<int>, CanalPrimario<int>, CanalSecundario<int>, Total<float>, Media<float>


Vemos que o *endpoint* `PixLiquidadosAtual` retorna 4 propriedades:

- `Data<datetime>`: data das operações
- `Quantidade<int>`: quantidade de operações realizadas na data
- `Total<float>`: financeiro das operações realizdas na data
- `Media<float>`: média das operações realizadas na data

As propriedades são atributos de objetos da classe `bcb.odata.api.Endpoint`, retornados pelo método `get_endpoint`.

In [4]:
ep = pix.get_endpoint("PixLiquidadosAtual")
ep.Data
ep.Media

<Property Media<float>>

Para acessar os dados deste *endpoint* é necessário executar uma `query` nesse objeto.

In [5]:
ep.query().limit(5).collect()

,Data,Quantidade,CanalPrimario,CanalSecundario,Total,Media
0,2025-05-15,203155891,202530064,625827,1.051084e+08,517.38
1,2024-04-19,163588000,163124269,463731,7.911428e+07,483.62
2,2024-05-24,156097661,155912971,184690,6.866069e+07,439.86
3,2024-02-10,140720681,140509596,211085,2.550603e+07,181.25
4,2025-11-18,200922887,200484305,438582,9.788716e+07,487.19


Ao realizar a `query` no *endpoint* limitamos a consulta a retornar 10 elementos, apenas para visualizar os dados da consulta. A consulta retorna um DataFrame pandas onde as colunas são as propriedades do *endpoint*.

Veremos abaixo, com mais detalhes, como realizar consultas nas APIs e quais os tipos de *endpoints* disponíveis (`EntitySets` e `FunctionImports`).

## Como Realizar Consultas em APIs OData

As consultas são realizadas através do método `query` da classe `bcb.odata.api.Endpoint`. Este método retorna um objeto `bcb.odata.framework.ODataQuery` que abstrai a consulta e permite executar algumas firulas como: filtros e ordenação. A classe `bcb.odata.framework.ODataQuery` tem os seguintes métodos:

- `bcb.odata.framework.ODataQuery.filter`: define filtros na consulta, com uma clausula `where` no SQL.
- `bcb.odata.framework.ODataQuery.select`: seleciona as propriedades retornadas pela consulta.
- `bcb.odata.framework.ODataQuery.orderby`: ordena a consulta pelas propriedades.
- `bcb.odata.framework.ODataQuery.limit`: limita os resultados a `n` registros.
- `bcb.odata.framework.ODataQuery.parameters`: *endpoints* do tipo `FunctionImports` possuem parâmetros que são definidos por este método.
- `bcb.odata.framework.ODataQuery.collect`: o *framework* tem uma abordagem *lazy*, dessa forma, este método realiza a consulta trazendo os dados e retornando um DataFrame.
- `bcb.odata.framework.ODataQuery.text`: este método retorna o texto (formato json) retornado pela API.
- `bcb.odata.framework.ODataQuery.show`: imprime a estrutura da consulta.

Os métodos `filter`, `select`, `orderby`, `limit` e `parameters` retornam o objeto `bcb.odata.framework.ODataQuery`, e isso permite a realização de chamadas aninhadas que compõem a consulta.

Por exemplo, na consulta do PIX, as datas não estão ordenadas, temos dias de 2021, 2022 e 2023 nos 10 registros retornados. Vamos ordernar pela propriedade `Data` de forma decrescente.

In [6]:
ep.query().orderby(ep.Data.desc()).limit(5).collect()

,Data,Quantidade,CanalPrimario,CanalSecundario,Total,Media
0,2026-11-24,229470668,228452399,1018269,1.209872e+08,527.24
1,2026-11-23,238874919,237602394,1272525,1.762647e+08,737.90
2,2026-11-22,178030801,177391578,639223,2.104139e+07,118.19
3,2026-11-21,236039166,235379720,659446,3.311344e+07,140.29
4,2026-11-20,233720205,232734942,985263,4.205198e+07,179.92


Veja que a consulta retorna as datas mais recentes primeiro.

Gosto de estruturar as consultas como uma *query* SQL. Sigamos com um exemplo:

``` SQL
select Data, Media
from PIX
where Data >= "2023-01-01"
order by Media desc
limit 10
```

Quero obter os 10 dias em 2023 que apresentam as maiores médias transacionadas no PIX.

Para executar essa query utilizo o método `select` passando as propriedades Data e Media, encadeio o método `filter` filtrando a propriedade Data maiores que 2023-01-01, e note que aqui utilizo um objeto `date`; objetos `datetime` também são aceitos. Na descrição do *endpoint* `PixLiquidadosAtual`, a propriedade Data aparece como `datetime` porque representa um campo OData `Edm.Date`. Sigo com o método `orderby` passando a propriedade média e indicando que a ordenação é decrescente e concluo com o método `limit` para obter os 10 primeiros registros. Na última linha executo o método `collect` que executa a consulta e retorna um DataFrame com os resultados.

In [7]:
from datetime import date
(ep.query()
    .select(ep.Data, ep.Media)
    .filter(ep.Data >= date(2023, 1, 1))
    .orderby(ep.Media.desc())
    .limit(5)
    .collect())

,Data,Media
0,2025-06-30,745.06
1,2026-08-31,740.18
2,2026-07-20,738.39
3,2026-11-23,737.90
4,2026-11-03,724.01


### Visualizando a Consulta

Algumas consultas podem ficar bastante complicadas, dependendo da quantidade de elementos que compõem a consulta. Para ajudar na construção e na depuração da *query*, criamos o método `show` imprime a query na tela, mas não a executa.

In [8]:
(ep.query()
    .select(ep.Data, ep.Media)
    .filter(ep.Data >= date(2023, 1, 1))
    .orderby(ep.Media.desc())
    .limit(5)
    .show())

URL:
  https://olinda.bcb.gov.br/olinda/servico/SPI/versao/v1/odata/PixLiquidadosAtual
Query Parameters:
  $format = json
  $filter = Data ge 2023-01-01
  $orderby = Media desc
  $select = Data,Media
  $top = 5
Return: Data<datetime>, Quantidade<int>, CanalPrimario<int>, CanalSecundario<int>, Total<float>, Media<float>


### Filtrando Dados

Os filtros são criados com o método `filter` e aplicados às propriedades do *endpoint*, por isso é necessário conhecê-lo, o que deve ser feito com o método `describe`.

In [9]:
from bcb import Expectativas
em = Expectativas()
em.describe('ExpectativasMercadoTop5Anuais')


EntitySet (Endpoint): ExpectativasMercadoTop5Anuais
EntityType: br.gov.bcb.olinda.servico.Expectativas.ExpectativasMercadoTop5Anual
Properties: Indicador<str>, Data<str>, DataReferencia<str>, tipoCalculo<str>, Media<float>, Mediana<float>, DesvioPadrao<float>, Minimo<float>, Maximo<float>


O *endpoint* `ExpectativasMercadoTop5Anuais` da API de Expectativas possui a propriedade `Indicador`, do tipo `str`. Vamos filtrar os dados com a propriedade `Indicador` igual a `IPCA`. Como o tipo dessa propriedade é `str`, utilizamos uma string no filtro e o operador `==`, que representa igualdade.

In [10]:
ep = em.get_endpoint('ExpectativasMercadoTop5Anuais')
query = ep.query().filter(ep.Indicador == 'IPCA').limit(5)
query.show()

URL:
  https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata/ExpectativasMercadoTop5Anuais
Query Parameters:
  $format = json
  $filter = Indicador eq 'IPCA'
  $top = 5
Return: Indicador<str>, Data<str>, DataReferencia<str>, tipoCalculo<str>, Media<float>, Mediana<float>, DesvioPadrao<float>, Minimo<float>, Maximo<float>


O método `show` apresenta os parâmetros da *query* formatados, com isso podemos visualizar como os parâmetros da consulta serão enviados à API. Note que o operador `==` foi convertido para `eq`. Podemos utilizar todos os operadores de comparação nos filtros.

In [11]:
query.collect()

,Indicador,Data,DataReferencia,tipoCalculo,Media,Mediana,DesvioPadrao,Minimo,Maximo
0,IPCA,2018-01-18,2019,L,4.13,4.13,0.13,4.00,4.25
1,IPCA,2018-01-18,2020,L,4.00,4.00,0.00,4.00,4.00
2,IPCA,2018-01-18,2021,L,3.94,4.00,0.11,3.75,4.00
3,IPCA,2018-01-18,2018,L,3.68,3.76,0.26,3.27,3.95
4,IPCA,2018-01-19,2019,C,4.13,4.25,0.25,3.70,4.30


Mais filtros podem ser adicionados ao método `filter`, e também podemos aninhar chamadas do método `filter`.

In [12]:
query = (ep.query()
           .filter(ep.Indicador == 'IPCA', ep.DataReferencia == 2023)
           .filter(ep.Data >= date(2022, 1, 1))
           .filter(ep.tipoCalculo == 'C')
           .limit(5))
query.show()
query.collect()

URL:
  https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata/ExpectativasMercadoTop5Anuais
Query Parameters:
  $format = json
  $filter = Indicador eq 'IPCA' and DataReferencia eq '2023' and Data ge '2022-01-01' and tipoCalculo eq 'C'
  $top = 5
Return: Indicador<str>, Data<str>, DataReferencia<str>, tipoCalculo<str>, Media<float>, Mediana<float>, DesvioPadrao<float>, Minimo<float>, Maximo<float>


,Indicador,Data,DataReferencia,tipoCalculo,Media,Mediana,DesvioPadrao,Minimo,Maximo
0,IPCA,2022-01-03,2023,C,3.1761,3.3000,0.4385,2.5,3.8000
1,IPCA,2022-01-04,2023,C,3.1761,3.3000,0.4385,2.5,3.8000
2,IPCA,2022-01-05,2023,C,3.1761,3.3000,0.4385,2.5,3.8000
3,IPCA,2022-01-06,2023,C,3.1761,3.3000,0.4385,2.5,3.8000
4,IPCA,2022-01-07,2023,C,3.3583,3.3302,0.5711,2.5,4.2693


Todos os filtros estão no atributo `$filter` da consulta e são concatenados com o operador booleano `and`.

Para combinar condições com `or`, use o operador `|` entre filtros. Use parênteses em cada comparação, pois `|` tem precedência diferente dos operadores de comparação do Python.

In [13]:
query = (ep.query()
           .filter((ep.Indicador == 'IPCA') | (ep.Indicador == 'IGP-M'))
           .filter(ep.DataReferencia == 2023)
           .limit(5))
query.show()

URL:
  https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata/ExpectativasMercadoTop5Anuais
Query Parameters:
  $format = json
  $filter = (Indicador eq 'IPCA' or Indicador eq 'IGP-M') and DataReferencia eq '2023'
  $top = 5
Return: Indicador<str>, Data<str>, DataReferencia<str>, tipoCalculo<str>, Media<float>, Mediana<float>, DesvioPadrao<float>, Minimo<float>, Maximo<float>


Também é possível combinar filtros explicitamente com `&`. O operador nativo `or` do Python não deve ser usado, porque ele avalia objetos em contexto booleano em vez de construir uma expressão OData.

É necessário conhecer o tipo da propriedade para saber como passar o objeto para a consulta. Os tipos de propriedade podem ser: str, float, int e datetime. Para propriedades OData `Edm.Date`, passe um objeto `datetime.date` ou `datetime.datetime` para o método `filter`; strings de data não são convertidas automaticamente pelo construtor de filtros.

In [14]:
ep = pix.get_endpoint("PixLiquidadosAtual")
(ep.query()
   .filter(ep.Data >= date(2023, 1, 1))
   .limit(5)
   .show())

URL:
  https://olinda.bcb.gov.br/olinda/servico/SPI/versao/v1/odata/PixLiquidadosAtual
Query Parameters:
  $format = json
  $filter = Data ge 2023-01-01
  $top = 5
Return: Data<datetime>, Quantidade<int>, CanalPrimario<int>, CanalSecundario<int>, Total<float>, Media<float>


O objeto `date` ou `datetime` é formatado como data na consulta; note que não há aspas na definição da data no filtro.

### Ordenando os Dados

A ordenação é definida no método `orderby` passando um objeto da classe `bcb.odata.framework.ODataPropertyOrderBy` que é obtida dos métodos `asc` e `desc` da propriedade.

In [15]:
ep = pix.get_endpoint("PixLiquidadosAtual")
ep.Data.asc()

<Data asc>

Este objeto é passado para o método `orderby` na *tripa* na qual a *query* é construída.

In [16]:
query = (ep.query()
           .orderby(ep.Data.asc())
           .limit(5))
query.show()
query.collect()

URL:
  https://olinda.bcb.gov.br/olinda/servico/SPI/versao/v1/odata/PixLiquidadosAtual
Query Parameters:
  $format = json
  $orderby = Data asc
  $top = 5
Return: Data<datetime>, Quantidade<int>, CanalPrimario<int>, CanalSecundario<int>, Total<float>, Media<float>


,Data,Quantidade,CanalPrimario,CanalSecundario,Total,Media
0,2020-11-03,2345,None,None,210.24,89.65
1,2020-11-04,2629,None,None,336.37,127.94
2,2020-11-05,16669,None,None,2685.84,161.13
3,2020-11-06,57936,None,None,21133.81,364.78
4,2020-11-07,11089,None,None,2333.89,210.47


O método `orderby` pode receber diversas propriedades para a definição da ordenação.

In [17]:
ep = em.get_endpoint('ExpectativasMercadoTop5Anuais')
query = (ep.query()
           .orderby(ep.Data.desc(), ep.Indicador.desc())
           .limit(5))
query.show()

URL:
  https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata/ExpectativasMercadoTop5Anuais
Query Parameters:
  $format = json
  $orderby = Data desc,Indicador desc
  $top = 5
Return: Indicador<str>, Data<str>, DataReferencia<str>, tipoCalculo<str>, Media<float>, Mediana<float>, DesvioPadrao<float>, Minimo<float>, Maximo<float>


Também podem ser realizadas chamadas aninhadas do método `orderby`.

In [18]:
query = (ep.query()
           .orderby(ep.Data.desc())
           .orderby(ep.Indicador.desc())
           .limit(5))
query.show()

URL:
  https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata/ExpectativasMercadoTop5Anuais
Query Parameters:
  $format = json
  $orderby = Data desc,Indicador desc
  $top = 5
Return: Indicador<str>, Data<str>, DataReferencia<str>, tipoCalculo<str>, Media<float>, Mediana<float>, DesvioPadrao<float>, Minimo<float>, Maximo<float>


Vejam que a consulta é exatamente a mesma.

### Selecionando as Propriedades

O médoto `select` funciona de forma muito semelhante ao *select* de uma *query* SQL.

In [19]:
ep = em.get_endpoint('ExpectativasMercadoTop5Anuais')
(ep.query()
   .select(ep.Indicador, ep.Data, ep.DataReferencia, ep.tipoCalculo, ep.Media)
   .orderby(ep.Data.desc())
   .limit(5)
   .collect())

,Indicador,Data,DataReferencia,tipoCalculo,Media
0,Câmbio,2026-11-19,2026,M,5.5926
1,Câmbio,2026-11-19,2027,M,5.5310
2,Câmbio,2026-11-19,2028,M,5.5073
3,Câmbio,2026-11-19,2029,M,5.5467
4,Câmbio,2026-11-19,2030,M,5.6100


Selecionar as colunas é importante para reduzir o volume de dados trafegado, pois a API do BCB não tem um bom desempenho, logo, essas configurações aceleram as consultas.

### Método `limit`

O método `limit` define a quantidade de linhas que será retornada pela consulta. Esse método é importante para investigar as consultas na API de forma rápida.

In [20]:
ep = pix.get_endpoint("PixLiquidadosAtual")
(ep.query()
   .filter(ep.Data >= date(2023, 1, 1))
   .limit(5)
   .collect())

,Data,Quantidade,CanalPrimario,CanalSecundario,Total,Media
0,2025-05-15,203155891,202530064,625827,1.051084e+08,517.38
1,2024-04-19,163588000,163124269,463731,7.911428e+07,483.62
2,2024-05-24,156097661,155912971,184690,6.866069e+07,439.86
3,2024-02-10,140720681,140509596,211085,2.550603e+07,181.25
4,2025-11-18,200922887,200484305,438582,9.788716e+07,487.19


### Tipos de *endpoints*

Como foi visto anteriormente, a API do PIX (`SPI`) possui 4 `EntitySets` e estes são os *endpoints* dessa API. Entretanto, há APIs que têm um outro tipo de *endpoint*, os `FunctionImports`. A API do PTAX, por exemplo

In [21]:
from bcb import PTAX

ptax = PTAX()
ptax.describe()

EntitySets:

EntitySet (Endpoint): Moedas
EntityType: br.gov.bcb.olinda.servico.PTAX.TipoMoeda
Properties: simbolo<str>, nomeFormatado<str>, tipoMoeda<str>
FunctionImports:

Function: CotacaoMoedaPeriodoFechamento
Parameters: codigoMoeda <str>, dataInicialCotacao <str>, dataFinalCotacao <str>
EntitySet: _CotacaoMoedaPeriodoFechamento
EntityType: br.gov.bcb.olinda.servico.PTAX.TipoCotacaoDolarAberturaOuIntermediario
Properties: cotacaoCompra <float>, cotacaoVenda <float>, dataHoraCotacao <str>, tipoBoletim <str>

Function: CotacaoMoedaAberturaOuIntermediario
Parameters: codigoMoeda <str>, dataCotacao <str>
EntitySet: _CotacaoMoedaAberturaOuIntermediario
EntityType: br.gov.bcb.olinda.servico.PTAX.TipoCotacaoDolarAberturaOuIntermediario
Properties: cotacaoCompra <float>, cotacaoVenda <float>, dataHoraCotacao <str>, tipoBoletim <str>

Function: CotacaoMoedaDia
Parameters: moeda <str>, dataCotacao <str>
EntitySet: _CotacaoMoedaDia
EntityType: br.gov.bcb.olinda.servico.PTAX.TipoCotacaoMoeda


Esta API tem 1 `EntitySet` e 6 `FunctionImports`. Assim como os `EntitySets`, os `FunctionImports` também retornam dados em formato tabular. A principal diferença entre os `EntitySets` e `FunctionImports` é que estes possuem parâmetros, como uma função, e estes parâmetros devem ser definidos para que a consulta seja realizada.

Vamos ver o *endpoint* `CotacaoMoedaPeriodo`

In [22]:
ptax.describe("CotacaoMoedaPeriodo")


Function: CotacaoMoedaPeriodo
Parameters: moeda <str>, dataInicial <str>, dataFinalCotacao <str>
EntitySet: _CotacaoMoedaPeriodo
EntityType: br.gov.bcb.olinda.servico.PTAX.TipoCotacaoMoeda
Properties: paridadeCompra <float>, paridadeVenda <float>, cotacaoCompra <float>, cotacaoVenda <float>, dataHoraCotacao <str>, tipoBoletim <str>


Este *endpoint* tem 3 parâmetros:

- codigoMoeda \<str\>
- dataInicial \<str\>
- dataFinalCotacao \<str\>

Para conhecer como os parâmetros devem ser definidos é necessário ler a documentação da API. Eventualmente a definição dos parâmetros não é óbvia. Os parâmetros de data da PTAX aceitam strings ISO (`YYYY-MM-DD`), `datetime.date`, `datetime.datetime` e `pandas.Timestamp`. Strings já no formato PTAX também continuam aceitas. A biblioteca converte os valores generalizados para o formato aceito pelo serviço PTAX: `MM-DD-YYYY` (mês/dia/ano, sem zero à esquerda).

Vamos realizar uma consulta para obter as cotações de dólar americano entre 2022-01-01 e 2022-01-05.

In [23]:
ep = ptax.get_endpoint("CotacaoMoedaPeriodo")
(ep.query()
   .parameters(moeda="USD",
               dataInicial="2022-01-01",
               dataFinalCotacao="2022-01-05")
   .collect())

,paridadeCompra,paridadeVenda,cotacaoCompra,cotacaoVenda,dataHoraCotacao,tipoBoletim
0,1.0,1.0,5.5884,5.5890,2022-01-03 10:04:22.186,Abertura
1,1.0,1.0,5.6242,5.6248,2022-01-03 11:11:42.883,Intermediário
2,1.0,1.0,5.6444,5.6450,2022-01-03 12:09:19.760,Intermediário
3,1.0,1.0,5.6643,5.6649,2022-01-03 13:11:50.353,Intermediário
4,1.0,1.0,5.6303,5.6309,2022-01-03 13:11:50.357,Fechamento
5,1.0,1.0,5.6902,5.6908,2022-01-04 10:05:22.015,Abertura
6,1.0,1.0,5.7015,5.7021,2022-01-04 11:05:20.148,Intermediário
7,1.0,1.0,5.6487,5.6493,2022-01-04 12:10:19.466,Intermediário
8,1.0,1.0,5.6674,5.6680,2022-01-04 13:08:59.118,Intermediário
9,1.0,1.0,5.6770,5.6776,2022-01-04 13:08:59.123,Fechamento


Note que a primeira data é 2022-01-03, pois os primeiros dias do ano não são úteis. Podemos aplicar filtros nessa consulta utilizando o método `filter`, da mesma forma que realizamos na consulta ao `EntitySet`.

In [24]:
(ep.query()
   .parameters(moeda="USD",
               dataInicial="2022-01-01",
               dataFinalCotacao="2022-01-05")
   .filter(ep.tipoBoletim == "Fechamento")
   .collect())

,paridadeCompra,paridadeVenda,cotacaoCompra,cotacaoVenda,dataHoraCotacao,tipoBoletim
0,1.0,1.0,5.6303,5.6309,2022-01-03 13:11:50.357,Fechamento
1,1.0,1.0,5.6770,5.6776,2022-01-04 13:08:59.123,Fechamento
2,1.0,1.0,5.6622,5.6628,2022-01-05 13:07:53.094,Fechamento


### Obtendo o Texto da API

Uma alternativa ao DataFrame retornado pela consulta, via o método `collect`, é obter o texto, em formato JSON (padrão) ou XML, retornado pela consulta.

O método `collect` faz o *parsing* do texto retornado na consulta e cria um DataFrame, o método `text` retorna esse texto bruto.

Isso é útil para fazer o armazenamento de dados da API para a construção de bancos de dados ou *data lakes*.

Para obter o conteúdo bruto, basta executar o método `text` ao invés do `collect`, ao fim da cadeia da consulta.

In [25]:
ep = em.get_endpoint('ExpectativasMercadoTop5Anuais')
(ep.query()
   .select(ep.Indicador, ep.Data, ep.DataReferencia, ep.tipoCalculo, ep.Media)
   .orderby(ep.Data.desc())
   .limit(5)
   .text())

'{"@odata.context":"https://was-p.bcnet.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata$metadata#ExpectativasMercadoTop5Anuais(Indicador,Data,DataReferencia,tipoCalculo,Media)","value":[{"Indicador":"Câmbio","Data":"2026-11-19","DataReferencia":"2026","tipoCalculo":"M","Media":5.5926},{"Indicador":"Câmbio","Data":"2026-11-19","DataReferencia":"2027","tipoCalculo":"M","Media":5.5310},{"Indicador":"Câmbio","Data":"2026-11-19","DataReferencia":"2028","tipoCalculo":"M","Media":5.5073},{"Indicador":"Câmbio","Data":"2026-11-19","DataReferencia":"2029","tipoCalculo":"M","Media":5.5467},{"Indicador":"Câmbio","Data":"2026-11-19","DataReferencia":"2030","tipoCalculo":"M","Media":5.6100}]}'

O texto retornado está no formato JSON. Contudo, as APIs OData também retornam conteúdo em XML. Para isso incluímos o método `format` na cadeia da consulta e passamos como parâmetro o tipo desejado.

In [26]:
ep = em.get_endpoint('ExpectativasMercadoTop5Anuais')
(ep.query()
   .select(ep.Indicador, ep.Data, ep.DataReferencia, ep.tipoCalculo, ep.Media)
   .orderby(ep.Data.desc())
   .format("xml")
   .limit(5)
   .text())

'<?xml version=\'1.0\' encoding=\'UTF-8\'?><a:feed xmlns:a="http://www.w3.org/2005/Atom" xmlns:m="http://docs.oasis-open.org/odata/ns/metadata" xmlns:d="http://docs.oasis-open.org/odata/ns/data" m:context="https://was-p.bcnet.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata$metadata#ExpectativasMercadoTop5Anuais(Indicador,Data,DataReferencia,tipoCalculo,Media)"><a:entry><a:title/><a:summary/><a:updated>2026-11-25T06:24:50Z</a:updated><a:author><a:name/></a:author><a:category scheme="http://docs.oasis-open.org/odata/ns/scheme" term="#br.gov.bcb.olinda.servico.Expectativas.ExpectativasMercadoTop5Anual"/><a:content type="application/xml"><m:properties><d:Indicador>Câmbio</d:Indicador><d:Data>2026-11-19</d:Data><d:DataReferencia>2026</d:DataReferencia><d:tipoCalculo>M</d:tipoCalculo><d:Media m:type="Decimal">5.5926</d:Media></m:properties></a:content></a:entry><a:entry><a:title/><a:summary/><a:updated>2026-11-25T06:24:50Z</a:updated><a:author><a:name/></a:author><a:category scheme="h

### O parâmetro `output='text'`

Para pipelines de dados onde é necessário persistir o dado bruto antes de qualquer transformação (camada SOR/SOT), o parâmetro `output='text'` pode ser passado diretamente ao método `collect` ou ao método `get` do endpoint. Isso evita serializar um DataFrame de volta para texto, o que pode ser uma operação com perda de informação.

``` python
ep = em.get_endpoint('ExpectativasMercadoTop5Anuais')

# via query chain
raw = (ep.query()
          .filter(ep.Indicador == 'IPCA')
          .limit(100)
          .collect(output='text'))

# via atalho get()
raw = ep.get(ep.Indicador == 'IPCA', limit=100, output='text')

# salvar em disco
with open('expectativas_raw.json', 'w') as f:
    f.write(raw)
```

O texto retornado é o JSON bruto da resposta OData, incluindo o campo `@odata.context` e o array `value`. O comportamento padrão (retorno de DataFrame) é mantido quando o parâmetro não é informado.

## Classe ODataAPI

O portal de Dados Abertos do Banco Central apresenta diversas APIs OData, são dezenas de APIs disponíveis. A URL com metadados de cada API pode ser obtida no [portal](https://dadosabertos.bcb.gov.br). A classe `bcb.odata.api.ODataAPI` permite acessar qualquer API OData de posse da sua URL.

Por exemplo, a API de estatísticas de operações registradas no Selic tem a seguinte URL:

    https://olinda.bcb.gov.br/olinda/servico/selic_operacoes/versao/v1/odata/

que pode ser obtida no portal de dados abertos no [link](https://dadosabertos.bcb.gov.br/dataset/estatisticas-selic-operacoes).

Essa API pode ser diretamente acessada através da classe `bcb.odata.api.ODataAPI`.

In [27]:
from bcb import ODataAPI
url = "https://olinda.bcb.gov.br/olinda/servico/selic_operacoes/versao/v1/odata/"
service = ODataAPI(url)
service.describe()

EntitySets:

EntitySet (Endpoint): DatasOperacoesSelic
EntityType: br.gov.bcb.olinda.servico.selic_operacoes.DataMov
Properties: DataMovimento<str>

EntitySet (Endpoint): OperacoesDisponiveis
EntityType: br.gov.bcb.olinda.servico.selic_operacoes.OperacoesDisponiveis
Properties: Codigo<str>, Categoria<str>, Descricao<str>

EntitySet (Endpoint): DataMesAnoUltimoMovimentoSelic
EntityType: br.gov.bcb.olinda.servico.selic_operacoes.DataUltimoMovimentoSelic
Properties: DataUltimoMovimento<str>, MesUltimoMovimento<str>, AnoUltimoMovimento<str>

EntitySet (Endpoint): OperacoesRegistradas
EntityType: br.gov.bcb.olinda.servico.selic_operacoes.OperacoesRegistradas
Properties: Codigo<str>, Categoria<str>
FunctionImports:

Function: OperacoesEmUmAno
Parameters: Ano <str>
EntitySet: _OperacoesEmUmAno
EntityType: br.gov.bcb.olinda.servico.selic_operacoes.OperacoesEmUmAno
Properties: Data <datetime>, Codigo <str>, Categoria <str>, Quantidade <int>, Valor <float>

Function: EvolucaoTipoOperacaoMensal
P

### Formatando parâmetros

Alguns *function imports* esperam parâmetros em um formato específico, por exemplo datas como `AAAAMM`. O argumento `parameter_formatters` associa o nome de um parâmetro a uma função, que recebe o valor informado na consulta e retorna o valor usado no literal OData. As aspas dos parâmetros `Edm.String` continuam sendo adicionadas pela biblioteca depois da formatação.

``` python
import pandas as pd
from bcb import ODataAPI

url = "https://olinda.bcb.gov.br/olinda/servico/IFDATA/versao/v1/odata/"
api = ODataAPI(
    url,
    parameter_formatters={"AnoMes": lambda v: v.strftime("%Y%m")},
)
ep = api.get_endpoint("IfDataCadastro")
ep.get(AnoMes=pd.Timestamp("2024-03-01"), limit=10)
```

O mesmo argumento é aceito pelas classes específicas, como `bcb.odata.api.PTAX`. Os formatadores informados são combinados com os definidos na classe (`PARAMETER_FORMATTERS`): é possível adicionar formatadores para outros parâmetros ou substituir os padrões sem criar uma subclasse e sem afetar outras instâncias.